## Repository analysis

In [ ]:
import json 
import pandas as pd 

In [ ]:
data = []
with open("../data/linux_kernel_init2.ctags", "r") as f: 
    for line in f: 
        data.append(json.loads(line))

print(data) 

In [ ]:
df = pd.DataFrame(data)

In [ ]:
df.drop(df[df._type != 'tag'].index, inplace=True)

In [ ]:
df[df.path == "../../linux/init/do_mounts.c"]

In [ ]:
!ls ../../../linux/init

### LLM API interactions

In [ ]:
def load_secrets(): 
    """
    Find our secrets and return them
    """
            
    hf_token = os.environ.get("HF_TOKEN")
    if not hf_token: 
        with open("../secrets/huggingface.token", "r") as file: 
            hf_token = file.read()
    
    openai_key = os.environ.get("OPENAI_API_KEY")
    if not openai_key: 
        with open("../secrets/openai.key", "r") as file: 
            openai_key = file.read() 
    
    return hf_token, openai_key 

In [ ]:
hf_token, openai_key = load_secrets()

In [ ]:
from openai import OpenAI

def run_completion(message, model="gpt-4.1-nano", api_key=None):
    """
    Push a chat completion request out to the void and collect the response ... we really need 
    to be able to run these in parallel given the volume of queries we have

    NOTE: with help from https://platform.openai.com/docs/guides/text?api-mode=chat&lang=python
    """
    
    client = OpenAI(
        api_key=api_key, 
        )

    completion = client.chat.completions.create(
        model=model,
        messages=[{"role": "user","content": message}]
    )

    response = completion.choices[0].message.content
    return response


In [ ]:
mini_response = run_completion("What kernel functions are implemented in the `init` subsystem of the Linux kernel?", model="gpt-4.1-mini", api_key=openai_key)
mini_response

In [ ]:
nano_response = run_completion("What kernel functions are implemented in the `init` subsystem of the Linux kernel?", model="gpt-4.1-nano", api_key=openai_key)
nano_response

In [ ]:
garbage = "It is desirable to augment a human practitioner in all of these tasks to mitigate the aforementioned risks and accelerate the acquisition of working knowledge. Large language models (LLMs) have displayed an exceptional ability to summarize, annotate, generalize and reason about source code. A suitably proficient model can serve as a human 'copilot', supporting analysis, summarization and general knowledge reinforcement. However, models are limited by the data and latent patterns provided during training. Codebases that were only superficially represented or absent entirely during the training phase present a problem when recruited into this copilot role."

In [ ]:
import nltk

In [ ]:

a = garbage
a = nano_response[0:100]

b = mini_response[0:100]
b = garbage


In [ ]:
import numpy as np
def sigmoid(x):
  """
  Calculates the sigmoid of the input x.

  Args:
    x: A NumPy array or scalar.

  Returns:
    A NumPy array or scalar representing the sigmoid of x.
  """
  return 1 / (1 + np.exp(-x))


In [ ]:
a_words = a.split(' ')
b_words = b.split(' ')
print(a_words) 
print(b_words) 
score = nltk.translate.bleu_score.sentence_bleu([a_words], b_words)
print(score)
print(sigmoid(score))


In [ ]:
import pandas as pd

In [ ]:
!ls

In [ ]:
datasets = [
    ('fs','../data/fs/fs.parquet'),
    ('ipc','../data/ipc/ipc.parquet'),
    ('init','../data/init/30june0116/init.parquet'),
    ('mm','../data/mm/mm.parquet') 
]

subsystems = [sub[0] for sub in datasets]
files = [sub[1] for sub in datasets]

In [ ]:
dfs = []
for dataset in datasets: 
    df = pd.read_parquet(dataset[1])
    df['subsystem'] = dataset[0]
    dfs.append(df)

linux_df = pd.concat(dfs)

In [ ]:
len(linux_df)

In [ ]:
linux_df.head()

In [ ]:
groups = linux_df.groupby('subsystem')

groups.head()

In [ ]:
groups.count()

In [ ]:
def stratify(x):
    print(x.columns)

In [ ]:
# Help from https://stackoverflow.com/questions/41035187/stratified-samples-from-pandas on sampling here: 
#stratified = groups.apply(lambda x: x.sample(frac=0.1, replace=False))
stratified = groups.apply(stratify)

In [ ]:
stratified

In [ ]:
stratified.count()

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
len(linux_df)

In [ ]:
linux_df.head()

In [ ]:
sub = linux_df['subsystem']
sub

In [ ]:
train, test, train2, test2 = train_test_split(linux_df, sub, stratify=sub, test_size=0.05, random_state=42)

In [ ]:
train

In [ ]:
train.subsystem.value_counts()

In [ ]:
test

In [ ]:
test.subsystem.value_counts()

In [ ]:
train2

In [ ]:
test2

In [ ]:
train.to_parquet("../data/linux/train.parquet")

In [ ]:
test.to_parquet("../data/linux/test.parquet")

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
a = [[0, 1], [0, 0]]
b = [[1, 1], [0,1]]

s = cosine_similarity(a,b)
np.diag(s)

In [ ]:
s[1][1]

In [ ]:
ipc_scores = [0.5711380046890133, 0.5339060327069189, 0.5208181217632747, 0.5666458502717515, 0.5950128249344271, 0.5707605837091287, 0.55823817475618, 0.5358225945427253, 0.5413236156851764, 0.5755969439570752, 0.5726276473824876, 0.5762706766362475, 0.5300458841124978, 0.5272494860490629, 0.5220369850853604, 0.5416233712590517, 0.5358755954079167, 0.5190797444537182, 0.5442946426761974, 0.4953554880884889, 0.5224802643890164, 0.5549338825687355, 0.5391955072431323, 0.5259685630388848, 0.5443190062087861, 0.49743721910085514, 0.5234162559491455, 0.5157855836377236, 0.47916189251605834, 0.5319198448850596, 0.5443190062087861, 0.528290725803098, 0.5069434053397687, 0.5477606441590511, 0.571423594475714, 0.5405963858743942, 0.5411041461557994, 0.5147289025578432, 0.5339637012150023, 0.5338572100044785, 0.4905496986554977, 0.5554350204760294, 0.5411041461557994, 0.522876882661518, 0.5099578462134375, 0.5525555184065835, 0.42789793452040503, 0.5460956784564515, 0.5411041461557994, 0.5006526288759464, 0.5318791147835816, 0.5177584174055317, 0.4406609725915005, 0.5289182323098436, 0.5480242551498373, 0.5136743040682521, 0.5474237005561039, 0.5511709213969511, 0.5450998031960236, 0.5536269670332128, 0.5493250769775067, 0.5503485635590919, 0.5272245993893466, 0.544039303701874, 0.5545241860043284, 0.5420121294098931, 0.5533390143322175, 0.5133514080604563, 0.5326673086389282, 0.5381631241301054, 0.5041670362868889, 0.5149781418604354, 0.5533390143322175, 0.5127427881425897, 0.5432694362039425, 0.5393573944912571, 0.555827094153319, 0.5196222755219156, 0.5349971761085001, 0.5338471032736636, 0.5275527493951486, 0.5392528937385368, 0.5266327900767757, 0.5206676802139663, 0.5552708993688692, 0.5629457876861355, 0.5545575989027484, 0.5392972477308965, 0.4432498349380313, 0.527956322365822, 0.4917836134552547, 0.4971244128723658, 0.4646539100439284, 0.5134556372113492, 0.5192661802415436, 0.49815879596074963, 0.5122574746282187, 0.5364015264327822, 0.5089102969483547, 0.5041621033455745, 0.5130250477619139, 0.5291356122168664, 0.523810380360532, 0.4966425668294849, 0.5302707410505455, 0.556910836381521, 0.539940830454193, 0.5359630396369685, 0.5089180453405604, 0.508504492685734, 0.47456259132783785, 0.4894139603602763, 0.50724860240754, 0.523578817470038, 0.5121750506041702, 0.4971466937965364, 0.520557852883602, 0.4862796227387079, 0.5157613393934275, 0.48332113799493154, 0.5301348570981326, 0.5240893902428594, 0.5104178703275897, 0.5511585224000353, 0.5285557799987151, 0.5460053804122769, 0.5003228038755192, 0.5168458020382467, 0.48956735879643254, 0.5167788998553057, 0.5065473939571619, 0.4712838446581236, 0.5296005083969775, 0.5195220313327649, 0.5229726403002688, 0.5475619974724699, 0.5536788801521876, 0.5431222443989239, 0.516450549629756, 0.5188427472847154, 0.4903081090817274, 0.49876631260270765, 0.5317659500127868, 0.5177922556083978, 0.507520149070116, 0.49832162807757874, 0.5094910037694418, 0.5375142113521858, 0.5385425017740577, 0.5136000119924572, 0.5395150612832311, 0.5075481288827091, 0.5415316670638968, 0.5352440278157671, 0.5436260440962126, 0.5399276968754656, 0.5198152762690773, 0.5098518935899125, 0.5072458725818335, 0.5288810370782987, 0.5005220472684342, 0.5404412052011945, 0.5312185967286179, 0.5524713424500594, 0.47177632580065554, 0.5446292707561645, 0.5381986510912425, 0.5514288452256224, 0.5092203370397325, 0.5311823473211, 0.510529689821611, 0.5494119880343159, 0.5418146394137087, 0.5042647445899292, 0.527394911095603, 0.5313723150049039, 0.5331591307693282, 0.506109491226784, 0.5538052872217951, 0.5442879237410725, 0.5641646786393946, 0.5381159639174108, 0.5161643127341411, 0.543509333824177, 0.9999999999999997, 0.5476890233491465, 0.5212515177316877, 0.5011450688554797, 0.5097558502914374, 0.524364939856713, 0.5086076253522727, 0.5374126834191583, 0.5501449482832858, 0.5714826101760283, 0.5672252501899598, 0.5721799324528751, 0.5878786238673647, 0.5783706413612625, 0.5721122126856121, 0.5776731742080684, 0.5854996380737459, 0.5912852076906603, 0.5970480801260594, 0.5890706750082942, 0.5977951184653414, 0.5401294535769039, 0.5524818083422318, 0.5953351462868053, 0.5742458017396688, 0.5556993069351589, 0.616065112038634, 0.6004291112977482, 0.6008533309602624, 0.603797779599716, 0.588454523329158, 0.5971221013341593, 0.5771850035611812, 0.5804219417352074, 0.5457400675612736, 0.5922745930382411, 0.5774698135143393, 0.5884305122824498, 0.5478413625644034, 0.5545334241971986, 0.48481995337993505, 0.5386748961729622, 0.5205388675441023, 0.5390885931303019, 0.5451024014070138, 0.5406113418846368, 0.5194784473661478, 0.531772108796097, 0.551274566420821, 0.5403967585095718, 0.5612644041465764, 0.5510961834789143, 0.5386625440569845, 0.5609524179895586, 0.5682775992573829, 0.5748420990939536, 0.4998553365346681, 0.49450546018559766, 0.4673996396748873, 0.4964606592250439, 0.5069799682593404, 0.5327550505771586, 0.5300979010708106, 0.5092908992066872, 0.507720568649303, 0.5564542464831043, 0.5230540467819832, 0.5097670628311475, 0.5045857071698758, 0.5260856197271178, 0.4978733787382765, 0.5146860249138858, 0.51042672819406, 0.493766906144962, 0.4981282192625449, 0.48942304107299484, 0.4882587021539456, 0.5302558337562475, 0.4974327664646522, 0.5048528283139433, 0.4914978730282037, 0.5029112675847117, 0.4794780855648374, 0.5084466893690307, 0.47723653252236214, 0.5252871977813105, 0.5630579482903978, 0.5036547870168366, 0.5190381158751156, 0.5059619016476984, 0.5648244621912295, 0.5151198860762463, 0.5445857211213593, 0.5458927312165237, 0.501306818559188, 0.5090985060653677, 0.546349433523565, 0.5124089014851523, 0.5289484993628172, 0.48546852677667385, 0.5075219323657816, 0.48909993361372445, 0.5016602487302453, 0.5317391485208588, 0.5289484993628172, 0.49298924305666936, 0.5186888957339507, 0.5206676577928352, 0.5117367005328826, 0.4958365003529667, 0.5084053582229906, 0.4780329758959321, 0.4996478554806284, 0.48883669790101164, 0.5013485351139995, 0.48611006096726095, 0.5148606847453847, 0.44831485723982134, 0.4915974238139495, 0.514155135428255, 0.4928608146099456, 0.4725795841296969, 0.5267820918189717, 0.48134656642926515, 0.490142130234722, 0.5138892636402865, 0.5222062838970316, 0.47830166155626885, 0.5244821126125286, 0.4939350692412681, 0.4980244210890361, 0.5093613099514656, 0.5077504970935494, 0.4525006588348982, 0.5244821126125286, 0.4787124331217308, 0.4871382755962921, 0.5169580152640195, 0.5059499906200662, 0.4662028394691543, 0.5244821126125286, 0.4907059146472961, 0.4805586861332942, 0.5213572236145304, 0.5168650810282314, 0.46785955474985585, 0.5244821126125286, 0.4812267969603403, 0.49316272698649094, 0.5224868979873509, 0.5138306744705534, 0.48648282751826283, 0.5244821126125286, 0.47220885808366747, 0.5066619209936611, 0.5143372884355488, 0.5156074680564106, 0.48284311376288536, 0.5271531710022601, 0.4666201038639129, 0.5088344862562645, 0.5197236642379648, 0.606426557249558, 0.4895340423296225, 0.5378472353260746, 0.49032597180932275, 0.4929321218064666, 0.532156944881458, 0.5254253010913699, 0.4782699672616132, 0.5410390175181977, 0.4701419308161699, 0.502510133513644, 0.4714342053344721, 0.5253789737488854, 0.4631612244215293, 0.5304990213139269, 0.5045042509637824, 0.5094531411940848, 0.5477520977434165, 0.5227619191168249, 0.490474463553641, 0.5086586239422534, 0.5054514984891225, 0.5094432675045342, 0.5364167693729409, 0.48204498719115413, 0.4806240868764364, 0.5304990213139269, 0.5029637550057479, 0.5028054231711786, 0.5279792228470557, 0.505531390619573, 0.5063464118283197, 0.5304990213139269, 0.5112713285139745, 0.5121159539516129, 0.5323562997323168, 0.5181987101547447, 0.49330839246625147, 0.5304990213139269, 0.5119011660855326, 0.5048845414633021, 0.5261063991873982, 0.49200061089282454, 0.4892818689126479, 0.5304990213139269, 0.5103844592548921, 0.5034989689853395, 0.5127419617800927, 0.5234410999049317, 0.5099296864709534, 0.5023842690554661, 0.48284248442892264, 0.48668305835261855, 0.49986119050778677, 0.4947937041781765, 0.4950977891133, 0.5156542268947466, 0.48304260083710676, 0.47964438811037113, 0.5081081939392771, 0.5127672497215667, 0.49604431858816167, 0.4892156349543441, 0.4764160881730298, 0.5082435589431223, 0.4876508786739804, 0.5002736407877383, 0.4939501167177448, 0.49480324900549444, 0.47547858215719013, 0.49573955544860626, 0.4956872767803254, 0.4873846753638216, 0.4740984190314864, 0.5156542268947466, 0.4954882764042932, 0.49099613752504223, 0.4723649472627132, 0.4928080104368683, 0.47422115080522803, 0.5138088252450196, 0.5279686837486399, 0.5163696150630251, 0.5099854606399807, 0.5058538792063085, 0.506388678967216, 0.5544415898911799, 0.5553366814706044, 0.49770457754775926, 0.5447667657063409, 0.5271933092757283, 0.5376461298885754, 0.5411399056110411, 0.528586976464634, 0.522669750936757, 0.5180046848820565, 0.5453684738577698, 0.5335935067168688, 0.5401794807234149, 0.5470994859809497, 0.5048981766312564, 0.4970202719906978, 0.49927559607917493, 0.5317907362932315, 0.5117152433087172, 0.5014125994073861, 0.5353267450799657, 0.5201905530098834, 0.49888994041998125, 0.5214618223966999, 0.5639166551837766, 0.5605691870576008, 0.5548065001039557, 0.5330679411619751, 0.5464326873342795, 0.5280192087699397, 0.554768013834847, 0.5331286981054306, 0.5678990211251242, 0.5322740825653909, 0.5765949806207342, 0.5167906683719322, 0.5241470403191644, 0.5624784582751184, 0.545047329159345, 0.5074555670023502, 0.5509910754525726, 0.531466627211189, 0.5141277606895476, 0.5260284169590714, 0.5283068144449636, 0.42167303787302285, 0.5135951882421854, 0.5461712794831908, 0.5199514261554599, 0.5131339384513561, 0.5287740176958509, 0.5397589460679564, 0.5674186057816237, 0.5116000162989786, 0.5298998915202182, 0.5327207961178264, 0.3993173069424648, 0.5150397607787682, 0.5096471897000505, 0.5159827454772323, 0.5037815968260984, 0.5069070407880721, 0.5297445834702337, 0.5105751034006573, 0.554418655483019, 0.5235242046869162, 0.5486670143560013, 0.5949647769527356, 0.5512895031531333, 0.5291262621349018, 0.5192279912410954, 0.5138282501788763, 0.5319593266854192, 0.5050694542578023, 0.5096594082753414, 0.5275998007863678, 0.5478384528997915, 0.5281627678914115, 0.5267609694700883, 0.5251517739315134, 0.5159674999900139, 0.5114692601773796, 0.5126184266996389, 0.5269260069393091, 0.5227144002606364, 0.5180871884355013, 0.5136617330882125, 0.5278700426866519, 0.5255709921813279, 0.5420570711713605, 0.5148808439242676, 0.5659861107859123, 0.5237876126868771, 0.46304299757346457, 0.49790864272187474, 0.5209880833976717, 0.49077490236883176, 0.5558166584924467, 0.5197343002591646, 0.5496162945525856, 0.4808882930687082, 0.5076925878892159, 0.49611385596682156, 0.549760877981256, 0.5120914027061719, 0.4854428310669086, 0.4847859174536411, 0.5221719947062896, 0.5214687520461967, 0.5555753432988457, 0.5354347692445812, 0.469536229044453, 0.49862018797596497, 0.504562610673992, 0.5391377127563632, 0.524607174284755, 0.5343315735854065, 0.46960856132602535, 0.5011305980397014, 0.524104576917695, 0.521447120508679, 0.5437665989393028, 0.5120385800121227, 0.5135097215467594, 0.5319609059498822, 0.5310716067181082, 0.5165545723064664, 0.5168291155870046, 0.5087499468042912, 0.5516155305288066, 0.5347236331370544, 0.5005703955426766, 0.5043443069276502, 0.49895901533715803, 0.5138805969194215, 0.4340234564317853, 0.5432505335604727, 0.5243853588643543, 0.5092304062877065, 0.5723411724764202, 0.5147874390279564, 0.5211797365934355, 0.44133104094546727, 0.5564558587786115, 0.48129306114016684, 0.4768571594814437, 0.5189363758196492, 0.5172114554051809, 0.5063029204109368, 0.449082343669397, 0.5601583844270677, 0.5745120116570055, 0.546875591600557, 0.4961924746582528, 0.5741054984232348, 0.5200705823335601, 0.5219557095580067, 0.5340877953000068, 0.5439722304102346, 0.5441056497254055, 0.5410793120747432, 0.5514057910477611, 0.5380961864678402, 0.5513850449855597, 0.5335628708289835, 0.5266690023477325, 0.5478131030182414, 0.5700646783431618, 0.5479484469352294, 0.5428203922755599, 0.5331086657630798, 0.5526949773953364, 0.5553522924730616, 0.5304349235346757, 0.5283648475048897, 0.5131075120690783, 0.543312996616293, 0.5251238527811501, 0.5394309646931961, 0.5463715217288809, 0.5339218899597249, 0.5111119636399798, 0.5205714302366093, 0.5408114448169237, 0.5380767639939716, 0.4986998946237641, 0.5474116046177409, 0.5413714856497127, 0.5563704323340282, 0.5309198704708659, 0.5482456015665811, 0.5525277149799606, 0.5768526486080339, 0.5411927496249647, 0.5519732284812704, 0.5073358990857426, 0.5395846211267418, 0.5583411985552972, 0.5708911858102634, 0.5260421388462644, 0.5820014181084928, 0.5436932320761391, 0.5302133252380028, 0.5281606195531499, 0.5876409077757645, 0.5372365427307897, 0.5864344966235099, 0.5785437644592373, 0.5433496029759901, 0.5537034533146348, 0.5791882112031298, 0.5535704560058385, 0.5536552987320049, 0.5661411661096618, 0.5263621168325344, 0.5202219185694368, 0.5717516932967197, 0.5649393407132046, 0.5877751364473975, 0.5816149131355639, 0.5556909410155028, 0.5550484779399352, 0.5644232527470431, 0.5329808874488067, 0.9999999999999997, 0.5385242444072932, 0.5325342505953151, 0.5582634563059443, 0.5914629266688407, 0.5428760349695398, 0.5635958188068175, 0.5379833339355976, 0.5599961256659458, 0.5690770823257225, 0.5858490191197079, 0.5551448280136804, 0.588205169860025, 0.5407652674233108, 0.562112377307224, 0.5684254348490393, 0.5671698393374343, 0.5509405076337355, 0.5140506707062245, 0.5569717716271481, 0.5264937262885787, 0.5425213875958854, 0.5303223801058581, 0.5426845728694178, 0.5414598006779054, 0.5405451372259036, 0.544279456115822, 0.5307692335500419, 0.5500936340034033, 0.5488023715990659, 0.5607494767015929, 0.5574192302549315, 0.5583557052438888, 0.5723309059999288, 0.5497885580811692, 0.5358097407591298, 0.553787086184695, 0.5626184846714707, 0.5513856450767194, 0.4698079640271653, 0.5181395282313785, 0.5031140837631733, 0.5548951650832552, 0.5093446495164535, 0.5065995483201718, 0.5261379679224736, 0.5610272947011077, 0.5449659740207273, 0.558435118448868, 0.5625431482398644, 0.5685975891229305, 0.9363579709623285, 0.5310604527386218, 0.5432574098575121, 0.5291005582277619, 0.5449522502586157, 0.5377856222111898, 0.5319575098684308, 0.5808977299413622, 0.5412153822097662, 0.5560647333502571, 0.5705731104213692, 0.5787755007108191, 0.5410538331157784, 0.5667727327345811, 0.5557730649820369, 0.5900146449558734, 0.4462422634039479, 0.5557149123069085, 0.5549864696084951, 0.5655107220354534, 0.5479419920840374, 0.5411444578046953, 0.5200521559021941, 0.5544039063086474, 0.5528438906782898, 0.4954748844149144, 0.526328692430259, 0.5081422612060027, 0.4971244406310069, 0.5300931920945251, 0.5552505999840917, 0.5092904540187005, 0.5477657091576889, 0.5130141397203241, 0.4944854781616145, 0.5294508485562696, 0.5379560194525798, 0.5447637685426449, 0.5423098475364255, 0.5504650395241542, 0.5224862006190671, 0.5287743922582251, 0.5581314525419547, 0.5352110109920044, 0.5531378866444229, 0.5211718557033516, 0.5221347032645853, 0.6039363360591736, 0.5450055459728187, 0.5359028204268219, 0.5594512271736308, 0.5140351231751247, 0.5090765294451416, 0.571772155839523, 0.49809071607825517, 0.5444684084736023, 0.565675592066858, 0.5539043059228161, 0.5410306577493029, 0.5798403581239061, 0.5752494203032684, 0.5457884999284832, 0.5826221289210278, 0.5360871116574021, 0.48532820421719935, 0.6128004837409726, 0.5769528614170827, 0.5362628301787172, 0.561016202029377, 0.5250189099691589, 0.5329742246470482, 0.5617937270483425, 0.5021250148592573, 0.5399493714013692, 0.551223602646427, 0.5185155337590536, 0.5526733985197998, 0.5784574246366738, 0.5520322535468394, 0.5401710825201888, 0.5607241674994241, 0.4973438317051343, 0.5121224548964336, 0.5752257531565442, 0.5528464276965609, 0.49844094338987605, 0.5299534172990249, 0.5110816951275167, 0.5202949516950091, 0.5541291545789742, 0.5182839576265221, 0.5170316909404382, 0.5117601734116237, 0.5170839195026927, 0.4679757577999538, 0.523495383430366, 0.5553730150738304, 0.5170316909404382, 0.503571503081371, 0.5082908077694535, 0.4843885077765525, 0.5696131710436456, 0.556016179615686, 0.5139310037507502, 0.527028696761363, 0.5200349810121546, 0.5265335224310472, 0.5496399457583154, 0.5403881959167567, 0.5157566982251502, 0.5262637106569473, 0.5371995846407985, 0.5154775884841054, 0.5325531384356786, 0.5467710092018598, 0.5287759048452974, 0.5617702016211631, 0.5594947485631079, 0.5405910027125711, 0.6044799682698738, 0.550541767969726, 0.5293596711348182, 0.5713706642933714, 0.548753031672518, 0.5218986873065845, 0.5687052807213075, 0.5631499056207653, 0.5365314684978791, 0.5483157650187791, 0.5190361272292681, 0.5519123755384505, 0.5741626415077098, 0.569601272140747, 0.525598881531722, 0.5509510944960891, 0.5270530803397638, 0.5076295141404448, 0.5867743743026231, 0.5563803072907042, 0.5252302624996865, 0.5590854150597203, 0.5327140394962095, 0.5024700071119985, 0.5901613135561561, 0.5412739587654278, 0.5251124061045359, 0.5388991619665358, 0.5245918669937262, 0.5314276651710705, 0.5630231139427946, 0.5621025336782521, 0.5219529122506805, 0.5318939293424209, 0.5322620649928311, 0.535464707450527, 0.5297770636835805, 0.5656406283101032, 0.5104900474706326, 0.519962480479871, 0.501281257510769, 0.4875700440283399, 0.5256202718495304, 0.500469501975565, 0.5389182615915847, 0.49565967977418146, 0.5226819118841016, 0.5102988381091991, 0.545686293799299, 0.4396515347283297, 0.4885560365462043, 0.5162225248088695, 0.496584241974709, 0.4866135727436426, 0.4792433592684917, 0.5217249875139136, 0.5197341218057197, 0.5200675217244484, 0.5273013075170816, 0.5448538455042121, 0.5317002604135643, 0.5069088417450697, 0.49720118299099925, 0.4726817737398038, 0.522415879416129, 0.47259356359231675, 0.5169315372672982, 0.5118903433733791, 0.4980435127727441, 0.49991508308059185, 0.5172411464707886, 0.5372123071502267, 0.5231853293743263, 0.49847428414031236, 0.49512696104743814, 0.49233621227878066, 0.5088605944809153, 0.5315544146892224, 0.525411007490244, 0.5009166609134663, 0.48324779426380826, 0.48725050446491075, 0.5039301047432826, 0.5060885892848445, 0.4938092732401099, 0.49499030450637, 0.5153321612806764, 0.5246395856270412, 0.5310424535957957, 0.5124980407140172, 0.5382359255167382, 0.5368164558240269, 0.5132798976686672, 0.5080880848988739, 0.5261647717008161, 0.5568335654735483, 0.5489141651739033, 0.5353975954365408, 0.5520503048746314, 0.5268644152912818, 0.548353624337821, 0.47302652827492925, 0.5762007539139634, 0.520051319066279, 0.5694515448545283, 0.5400995012523415, 0.5620230681326646, 0.5427367354303699, 0.554133085803385, 0.5044470375216505, 0.5123095855294455, 0.5201418506306039, 0.4984662901290488, 0.5280779059208003, 0.539604651607527, 0.4778094610068578, 0.5290815864455648, 0.5495722862476553, 0.5433285220611814, 0.5291252801303699, 0.5577598463142532, 0.5316144734916275, 0.5200551667475206, 0.5444154332973321, 0.5337266563813277, 0.5183732316801498, 0.5380427014448297, 0.5260436184604207, 0.5307521830157708, 0.5530178817099891, 0.5439583446182474, 0.5328247635001244, 0.5453979034935392, 0.5678476814245295, 0.5500826497666358, 0.5161749945746226, 0.5361562471309415, 0.4957447304106529, 0.4796587064721052, 0.5213430203502563, 0.5299698720746304, 0.5421674966282103, 0.5055534987369289, 0.49752636485817336, 0.517063519366424, 0.5308140675393694, 0.5136410116424417, 0.5370967737383656, 0.49711174771948763, 0.47665756768948453, 0.5238310373214862, 0.5482870852338928, 0.49926777590208077, 0.5517945487179424, 0.4805935055397177, 0.9094013674523169, 0.5599079724652445, 0.5202636650488545, 0.5223063317018424, 0.5341614095106297, 0.508934984799187, 0.4899123577226236, 0.5500723515578549, 0.5352477922400531, 0.5178526442105191, 0.533483332844154, 0.510696814576908, 0.5144730421025154, 0.5163064059586465, 0.5362493039198368, 0.5206311315571746, 0.4969341172966109, 0.4806274608434285, 0.49923197901603267, 0.49681621327453407, 0.5039484768272038, 0.5021601197561626, 0.5221717480641013, 0.5033856856960517, 0.5184296176756099, 0.5408548189789351, 0.5136715188005083, 0.49935300999432475, 0.5028259600657454, 0.4960498617374115, 0.4876302019486317, 0.48868633584539745, 0.5426450546320276, 0.5420465197730155, 0.4959992991090218, 0.5151514972290117, 0.5318589234231705, 0.538053750016105, 0.4992123501188362, 0.5253413108162316, 0.525979926979387, 0.5330364314126719, 0.4954003250074143, 0.5121919110315396, 0.5512989886621362, 0.521681901644905, 0.5241072667192419, 0.5312008958758492, 0.5165207456173204, 0.5885447844702275, 0.5437274710303971, 0.4781628271476263, 0.4869496256339041, 0.494189881357052, 0.49199779028996415, 0.5472977417575198, 0.5048834422929847, 0.4887321541353061, 0.5302749334492084, 0.477602482797265, 0.5088330091634742, 0.5283302867795987, 0.5315706688479797, 0.4877246537946312, 0.5083973082852147, 0.5056318223888543, 0.47980492112843387, 0.5409356556680603, 0.5286794882736752, 0.4985483221477941, 0.5067611844500627, 0.4964021788834343, 0.47274887070930327, 0.5013724320839681, 0.5370016241425194, 0.5215503939101682, 0.5603805372023462, 0.5333317154746, 0.5556634349118543, 0.5319061715617256, 0.5044894502164119, 0.5074923633408632, 0.5102840031727192, 0.5098260561734269, 0.5331783900626434, 0.5227969126126657, 0.5123702077130695, 0.535512943891197, 0.5284714337874012, 0.5298517585209797, 0.5610693835047232, 0.5293900734798, 0.5284199967044868, 0.5252270673118228, 0.5190105485947182, 0.5278593667153895, 0.5539923979567559, 0.5241558989097418, 0.518417275736305, 0.5536483806145803, 0.5088261234285826, 0.5430618687231148, 0.546530296379468, 0.5275507914588139, 0.5448558480553731, 0.5376451765366127, 0.544770025620597, 0.5208659002977232, 0.5529462723534311, 0.5584525381150371, 0.5633128850170237, 0.5494408480825854, 0.5420894011602725, 0.5119600397353757, 0.5349391145305575, 0.576574569017615, 0.5392469807105522, 0.5116562179603266, 0.5460023148949136, 0.5408205150043669, 0.48526027356661305, 0.575739535192646, 0.5384992229557426, 0.5247643794272407, 0.5101256318815232, 0.5261798234844747, 0.5137741801318432, 0.5125836211327284, 0.504659715792242, 0.536670993098809, 0.5462119520336876, 0.5031594345165215, 0.5667768826033851, 0.5349000267881897, 0.44278241164128335, 0.4999516151727851, 0.5203897110435507, 0.5098899847757419, 0.4719421883608949, 0.500537506478353, 0.5010068330519807, 0.5226276433206543, 0.48068608188510475, 0.5185393310130035, 0.5141041711589365, 0.5439926517103146, 0.4707001269097451, 0.50905269310054, 0.4625361894113838, 0.5067437333181577, 0.518197338077223, 0.48329651892668146, 0.4834024066599097, 0.514199348090623, 0.5190897441402544, 0.5377041705424885, 0.5245136236625372, 0.5643589650223171, 0.46207231994910997, 0.5041366328824006, 0.5106086994929768, 0.50165127714992, 0.4839437097361168, 0.4897644652691119, 0.4921876005620091, 0.5118780367119627, 0.530755405974056, 0.52847815458782, 0.5072870709133612, 0.5461349759045694, 0.5048518454620416, 0.5182249483954501, 0.5270213777141965, 0.537667543194466, 0.4916632710121592, 0.5764636746302884, 0.50923759668, 0.5191353608294205, 0.5328554917822121, 0.5133912266620059, 0.5560949406998394, 0.5352123350620153, 0.5121598541270672, 0.5155136677864207, 0.5423145387774342, 0.5436469610648424, 0.5421303267341493, 0.5327035746914774, 0.5415235151273435, 0.4930398855864482, 0.48338974905646753, 0.5116986865246484, 0.5050727435546762, 0.5293728078077982, 0.493179325787967, 0.5304089032650973, 0.501187807230808, 0.5363016032267953, 0.4272773710745544, 0.5082434706829244, 0.535270003677929, 0.5195922189925508, 0.5295078924173434, 0.5164242919753979, 0.5135343127051439, 0.5261947952928882, 0.5312677825100875, 0.5056713936713668, 0.50434238105483, 0.508909176969759, 0.5219128638973711, 0.4993518777992716, 0.529069854524948, 0.5109866614562426, 0.5001787740319381, 0.5178042693998931, 0.5088609255745301, 0.5050190842114337, 0.5240127709013176, 0.49097885413123243, 0.49517435632763207, 0.4726910451182167, 0.48018192452359515, 0.514294765947457, 0.5135719916420439, 0.5315988239394361, 0.5008504466399749, 0.5101445055570619, 0.5203833214784384, 0.5072052262187441, 0.5331614948047608, 0.5275292366873483, 0.5136066701754045, 0.508860339728747, 0.47856842560691604, 0.5411783925854775, 0.5183493345194017, 0.5090548642387265, 0.4969847082198545, 0.5047551211642003, 0.5267767632608215, 0.5197087774852651, 0.5159460706011237, 0.48410508036195354, 0.5150417725391766, 0.495932804618062, 0.5105097743628193, 0.5331712149378356, 0.5242378691172811, 0.506694960584687, 0.49732539461188874, 0.4798995955570227, 0.5077890160383941, 0.4916752233590087, 0.4909077948513771, 0.5121887812529589, 0.4955641587174481, 0.4736622022956329, 0.44353051623057294, 0.5354284676252763, 0.5012012883712677, 0.5161316198337554, 0.4894935075995766, 0.4971451201033899, 0.4914858512814369, 0.5155616788871167, 0.5048058308506431, 0.4928469398302908, 0.5117107752447194, 0.48291496066067935, 0.4945311352813025, 0.4974896994628879, 0.5159916132689703, 0.49743432525353387, 0.5133474506954429, 0.4996583511445809, 0.5411416508697284, 0.5500830435417982, 0.5225887306053428, 0.5066629575383912, 0.49643016751646335, 0.5062826901837625, 0.5077267624582114, 0.5191237182016939, 0.4994635297850003, 0.5061027723453068, 0.47527082320839104, 0.49056105330074345, 0.5218557038953688, 0.5298800510902008, 0.5076546044960286, 0.5336618815234315, 0.5160585190979239, 0.5124124750863195, 0.5233081924187302, 0.5439485318297951, 0.5251239370771873, 0.5263601865208429, 0.4906619148965576, 0.507842019355703, 0.5401099617816527, 0.49141174040664515, 0.5348472668085797, 0.521960518627787, 0.5015388266504976, 0.4998094401078296, 0.5296532187130549, 0.5654370356473846, 0.5200857141952574, 0.5463323464327128, 0.5350852769753693, 0.4956731337092349, 0.54214317258345, 0.5297681978602231, 0.498992531945388, 0.5098394437526383, 0.5337709555925987, 0.5240787547203842, 0.5439779225299579, 0.5350824028029453, 0.47660429085454353, 0.5221665102664769, 0.5274326091560682, 0.49048119794904754, 0.5070468424089685, 0.5391246588467156, 0.49366879638406935, 0.5231397123323634, 0.5192236956130407, 0.5161842895818317, 0.4804284756308267, 0.5284464517161219, 0.5677091034365588, 0.5117018057365839, 0.49993566598820777, 0.507937605842308, 0.4994344773187434, 0.525633642720827, 0.5155210728889696, 0.5330766338480902, 0.5207916552588006, 0.5184975047469818, 0.5300216240410214, 0.5285963079471909, 0.5267303761160494, 0.5105087931703765, 0.5372408082688531, 0.48646024894757456, 0.522080870952086, 0.46689171249607264, 0.5258051240233494, 0.5070867701959989, 0.47791417363756705, 0.5091233732367303, 0.5345048801919479, 0.5264280617844816, 0.507994212900668, 0.49690337352487013, 0.5072503250382294, 0.5024714993166617, 0.5340888123211738, 0.5122928033221323, 0.511432706292668, 0.5211888537723008, 0.49054511030079667, 0.5132140533936179, 0.5339650228124144, 0.5277174705251518, 0.5093124867833919, 0.5156287354257116, 0.4836682208644882, 0.48360681160700236, 0.4823849224587582, 0.4907570454798439, 0.5003066499746237, 0.5005333598329849, 0.4893287280974258, 0.516067306507038, 0.49131014008938984, 0.4840980017942152, 0.5111887885289261, 0.5179783547301856, 0.49212775580759627, 0.5077800421356544, 0.4838431119769478, 0.478472489603517, 0.5333673336610543, 0.4897574155013718, 0.46933627061576644, 0.5023427274786755, 0.44876547191470584, 0.5121993811832551, 0.48730943478335503, 0.4931257274030055, 0.5098440623597232, 0.4990538368786009, 0.5318637959383719, 0.5021281049175648, 0.5363217323133269, 0.5522053110299543, 0.4732340898536755, 0.5173857775401759, 0.5199733148110379, 0.5238791525427213, 0.5174079294802022, 0.47288929925892814, 0.4859977296585541, 0.47791933088973193, 0.49904596822841774, 0.49214231770851924, 0.5144163252423518, 0.5142619381665634, 0.5156646888002621, 0.5100078892612397, 0.4442226528656221, 0.5470389396956064, 0.5584124576113179, 0.5058592218579976, 0.4987618526043701, 0.45493309878741434, 0.48904626323492667, 0.5034196336149541, 0.5086681565918829, 0.533935292948994, 0.5297897044591391, 0.48861006306085175, 0.5104028522778675, 0.48656082812677903, 0.47822924851122733, 0.49081715299567563, 0.4986244002914691, 0.5246151028246389, 0.5052695527420414, 0.48597335621263943, 0.5045689408079791, 0.516045201043595, 0.46441801466038524, 0.5062341546415807, 0.4899771873168045, 0.47477959340447884, 0.5208130922855013, 0.49755961557449885, 0.48920167133884473, 0.4850647142726057, 0.4745291816975762, 0.5057825022804256, 0.49642551257331863, 0.5271437282817214, 0.49164958615485277, 0.5255443394085249, 0.4468570829697101, 0.49350338238129215, 0.5100811735180835, 0.5069312686120913, 0.5137156533609687, 0.5130828176490869, 0.5464634494961552, 0.4982881044745326, 0.5137978842164814, 0.5434742368616763, 0.4970147193819592, 0.5265221517448975, 0.5283474324063772, 0.5394745981627025, 0.5426346688625218, 0.5109777083580034, 0.4956535571377941, 0.4959484810352197, 0.49472955782159933, 0.5456568931056988, 0.507027052022911, 0.5088892960701437, 0.5285825296266491, 0.49013153197877657, 0.5316342183749341, 0.5281277532081099, 0.5019073410477954, 0.5092020612343271, 0.49446499609009975, 0.4792101797140189, 0.4723706547924755, 0.47785346021302433, 0.5120348048258112, 0.5288531271989146, 0.5171242938613223, 0.5082927861566218, 0.5553737775367749, 0.5314596408114931, 0.4922322841613676, 0.49505189977653985, 0.4774472118646121, 0.5033653033212929, 0.48419619221133187, 0.487021159675121, 0.509386879974636, 0.51655178024356, 0.4943049466153192, 0.5187149747394029, 0.5611284586552436, 0.4936668349216442, 0.5105731959042271, 0.5389689775590611, 0.5125681921443148, 0.5205596961876813, 0.5304519087692643, 0.507165721382921, 0.5231498058933464, 0.5505865264665393, 0.542437572597787, 0.5359117845571892, 0.5122873824341028, 0.5390289947260684, 0.5213294961300263, 0.5167177969122342, 0.5165660205785212, 0.4977498867006509, 0.49176762195594825, 0.5361643476129284, 0.5326533056420313, 0.5174338457190175, 0.522873312390072, 0.5091712419120387, 0.5170527615595357, 0.5327075304099568, 0.5007305039298746, 0.5271732555100748, 0.4948630257537076, 0.5151099818667504, 0.5417169038205775, 0.531106026666468, 0.5182677264374815, 0.500234351850984, 0.5432183200576729, 0.5239443246118245, 0.5038287349055085, 0.5545542604401519, 0.531785276134274, 0.5053656193297501, 0.520737088784962, 0.49209859912914944, 0.42159674388117957, 0.5379321007395503, 0.544574293594699, 0.479956775423095, 0.502697308471695, 0.4925592369179252, 0.5033302899912726, 0.49551758784845656, 0.4873392916384648, 0.48199361720091843, 0.48165649299034596, 0.5246984888050509, 0.5101355581803675, 0.499954992762018, 0.5100612372238531, 0.527464621023571, 0.5203721341685783, 0.49356127645105174, 0.5639046061778403, 0.5680581097419768, 0.5119627727698723, 0.5351751196803345, 0.5159107667013152, 0.517049269760222, 0.5760580629647432, 0.5554944788700784, 0.5132749472818499, 0.5240482647472215, 0.5081595483932826, 0.5386754737427054, 0.5735836688370035, 0.5293746297327244, 0.5375882268588654, 0.5640639744826348, 0.5194024102878618, 0.5302413190331201, 0.4964389952683184, 0.5142541415126867, 0.5038425566910476, 0.5571735917973871, 0.5133881521666521, 0.5559825974906529, 0.5071182105766644, 0.5340064026902253, 0.5243413464908425, 0.55740534924788, 0.4623246816314528, 0.5552824936249975, 0.5701698389845378, 0.5155368798677293, 0.5165763745465247, 0.5765148441526906, 0.5391771237835078, 0.5620192699465142, 0.5747686927972726, 0.5499309294212471, 0.5151226984157715, 0.5648294736885727, 0.5190489605057379, 0.5353679721650937, 0.5262402459182638, 0.5226312218101233, 0.4876467902862012, 0.5844192899588704, 0.5516376609267188, 0.5686342783133019, 0.546921494771433, 0.5513791902354586, 0.5348081143326507, 0.5739578007363993, 0.5100886540479975, 0.5567477693947243, 0.5257521260259092, 0.49565627979658045, 0.5243059736123258, 0.5874605077713251, 0.5467073443339746, 0.5520593871418074, 0.5324614330249433, 0.5260451944131717, 0.527267028681906, 0.5882957562097207, 0.5639377899780141, 0.5730473100590486, 0.5316210900458085, 0.5411474804976858, 0.5166741858628608, 0.5936076176746342, 0.5456963540986901, 0.5671757752394986, 0.5579122852477674, 0.5451565888291086, 0.5604872519204578, 0.543737698237744, 0.5145023738861775, 0.5305079763807463, 0.5294436545201968, 0.5631898242529725, 0.527923698742767, 0.5830607772796972, 0.5319588458484875, 0.5594657941677013, 0.5281676280583762, 0.5637108264960986, 0.5304695734559142, 0.5748360306624283, 0.5138551575317672, 0.5592348740121111, 0.48343253051823004, 0.5259608246971501, 0.49798135004306526, 0.5043757937925903, 0.5092848961140796, 0.5374136564915323, 0.5429439447352602, 0.5173658338647674, 0.5130862989237748, 0.528439375026849, 0.49218629964426563, 0.5204347359854715, 0.5289836416355207, 0.4967381881439391, 0.515620383313197, 0.5250603719986853, 0.5231768924501912, 0.49552327941224195, 0.5093908966670918, 0.5002344068142858, 0.5338813433362355, 0.529181241534904, 0.5539731305034905, 0.5207322325740578, 0.5719438931906797, 0.5292668696461929, 0.5117115474243864, 0.5464917711880181, 0.5505160858911006, 0.5522800621361814, 0.5480437990146644, 0.5266537008506114, 0.5159957921149682, 0.5679449623691568, 0.5488048121355105, 0.5191776926650651, 0.5770344330694943, 0.5453042491647537, 0.5588031876885755, 0.5679449623691568, 0.5337348373846851, 0.5382479062343468, 0.513467630549937, 0.5572022465307308, 0.5390320650380618, 0.5302986258535347, 0.49209362966942916, 0.5038791988499179, 0.4893861802198063, 0.5662176121463278, 0.5220029299027099, 0.5156754231490859, 0.4522709980356681, 0.5292163649859128, 0.5394608642794069, 0.5308240611531488, 0.5444279466525024, 0.5285207307394483, 0.5360817981904997, 0.5123316412336516, 0.5327600575887427, 0.5305730415874895, 0.5460498280924393, 0.5613712514663807, 0.45349132524811386, 0.5278248430421018, 0.5386060428425972, 0.5310291920406641, 0.5422709817041582, 0.5224313210070142, 0.5509945111990442, 0.5066998430095023, 0.5331301228586586, 0.5441908578095165, 0.5379291326467632, 0.5507284636841925, 0.568694943255605, 0.5209552752411406, 0.527084372255872, 0.5130721974349166, 0.5533997562964065, 0.5447844621061282, 0.5457593881602463, 0.5255540894306927, 0.5499350731340192, 0.57252192535693, 0.537596543816208, 0.5272289190124522, 0.5158659274295465, 0.49146688456647564, 0.4843271386804092, 0.532178910164139, 0.4695305068977609, 0.5849385871361519, 0.5379128041214183, 0.5380656201211054, 0.5246346992672634, 0.5319394639983179, 0.5390172923221603, 0.49601331510024693, 0.49214920398630113, 0.5043260409648289, 0.5146053336930874, 0.5245081977867274, 0.5359875634295329, 0.5155489147908925, 0.4990892598622132, 0.4986184239234505, 0.5501761351974384, 0.5154077838483897, 0.5271404146660796, 0.5096466436685825, 0.54015194646859, 0.5105015431884672, 0.497869211790653, 0.5415126417309486, 0.529159045707885, 0.5254772203757667, 0.490399051412395, 0.5413617868248751, 0.535022154375479, 0.5397437745906553, 0.47452588095502324, 0.5158105374274627, 0.5064037488302153, 0.49211240975519455, 0.5398327062300217, 0.5358634132569037, 0.5015045035108644, 0.5398993022548522, 0.5090771552992341, 0.5242622820175733, 0.4980237485488014, 0.5222205330150862, 0.521485760203007, 0.5278507921783586, 0.505640780644665, 0.5234374648119839, 0.5229275668082868, 0.5203889356975123, 0.5122345388661531, 0.5167486511331916, 0.4937542514490342, 0.5002577462059392, 0.5240070810855177, 0.5362802999516918, 0.4918777567178756, 0.5035210067466889, 0.5007614366827533, 0.5072770746110777, 0.5129926210724285, 0.4893552220159438, 0.5286567495979533, 0.527207017019021, 0.5477937522797365, 0.5541763446401132, 0.4576091170235963, 0.5351747326421332, 0.5311123997992554, 0.5339297806302934, 0.5348846766420272, 0.48731017066713966, 0.551115432945193, 0.5491357576524665, 0.49807086501750164, 0.5468387280787163, 0.5538056220945328, 0.5260330234712542, 0.5405396589142766, 0.5592760307510913, 0.5485424837787303, 0.5179572580885238, 0.5262883834514276, 0.5241897595633347, 0.5589842116997934, 0.5357581509907783, 0.5458120952319244, 0.5203122252228608, 0.5378255327426354, 0.49795344769019156, 0.5570205090576026, 0.5397312153535099, 0.5444421109111448, 0.5537306755983867, 0.5459807574488639, 0.5297671663896423, 0.58137432799541, 0.5625916140510463, 0.5426850101313159, 0.5537306755983867, 0.5489429126213021, 0.5058391938273713, 0.4907109171017283, 0.561969633726896, 0.5108191079891156, 0.5433882582361749, 0.5105867971424192, 0.5421262284125601, 0.5363571011561004, 0.5663089298406501, 0.5378501920746972, 0.5273602069799243, 0.551612217010999, 0.5161460588969463, 0.5504892396254419, 0.5466653909519912, 0.5190585120528898, 0.5274232259936122, 0.5540134528789711, 0.5421750057255454, 0.5461611131899717, 0.5364555191969834, 0.5256183165388286, 0.5138886374054241, 0.5585687487061255, 0.5434410349066419, 0.5415344066929928, 0.5671154010297865, 0.542337148710297, 0.5071387815840622, 0.5259381547918474, 0.5189048524695343, 0.46819262862328703, 0.5510946042852052, 0.5219805774932922, 0.5201818407065331, 0.5289918293575083, 0.5011903529570945, 0.5094699984404005, 0.5773736991737443, 0.5415387302452341, 0.5234400719661813, 0.4948973664845364, 0.48797293594010926, 0.5440310324131383, 0.5412706137573102, 0.5222448686676161, 0.4895162848845091, 0.5318796618723319, 0.49953135780989366, 0.5212626880330123, 0.5795992399628078, 0.5398385273854411, 0.5265660347013044, 0.5315814750992823, 0.5281225394066666, 0.5233723151751858, 0.5362793323269308, 0.5334883623167839, 0.5232377657208475, 0.5555363610466071, 0.5149845684995142, 0.5349321415609063, 0.5601767532567299, 0.50625614203579, 0.5282497306421243, 0.49834152483537536, 0.5132865724312754, 0.5582136190405866, 0.5518607904180842, 0.5082085733936669, 0.526572063421048, 0.48773893415935543, 0.49826885330867166, 0.5303899383570427, 0.5517863812011443, 0.5103489028288579, 0.5414710667691673, 0.5573829070654086, 0.5515903617875473, 0.4683675331758043, 0.5681295343241035, 0.5619066181968558, 0.532902351155762, 0.5419481571669883, 0.5211711668834477, 0.540574516510195, 0.5686769934888869, 0.5419371594665351, 0.5144833838952096, 0.4970922226031385, 0.49982428251981703, 0.4785850604344045, 0.5462637634997516, 0.5454162335610071, 0.532795609923101, 0.5515552042648972, 0.530101757427754, 0.4821759500477996, 0.557390940846265, 0.5318452464194703, 0.5425212184350837, 0.525840517147231, 0.5044451445486935, 0.4756404497410178, 0.5349748590121582, 0.5128626656068984, 0.5397158220608423, 0.5392439029618789, 0.5465451977708862, 0.5427473181979505, 0.556452581592316, 0.5434808104601726, 0.5067703066637186, 0.5272517183723905, 0.5074506141163182, 0.5081841285511028, 0.5599448115637633, 0.5488754208116696, 0.5641829563136439, 0.5540161133896868, 0.5276071457124409, 0.559749642090346, 0.5533694186053768, 0.5407864444900127, 0.5532005894172101, 0.5335963634978479, 0.5280387154828126, 0.5071430488947082, 0.48347521905294283, 0.5147532173074518, 0.5032520204465132, 0.5207333936021381, 0.5145865886831216, 0.5023404644103919, 0.5474564760451422, 0.5276699094675814, 0.514441542718938, 0.5075043116538457, 0.5077982032612548, 0.5378290429414482, 0.5190116573528281, 0.5211175497936087, 0.5594548483266852, 0.5266256696712857, 0.5139398833726756, 0.5314016853582431, 0.5628220258505685, 0.5312968240094305, 0.5440561986966277, 0.5005611731976937, 0.5366856339411048, 0.5164197147640556, 0.4999063444072427, 0.484544585516095, 0.5088510930940577, 0.5097119477587757, 0.5080990004182737, 0.5040147716054663, 0.5469589784752111, 0.5341407889464731, 0.518012853863016, 0.4969108480197177, 0.47992735784004137, 0.511520170486922, 0.5334659727808371, 0.5304824333249736, 0.5314762069699066, 0.5232676778658893, 0.531324080246199, 0.5275162041111856, 0.5222620506196871, 0.5374803204317233, 0.5170259174959573, 0.5474297280765286, 0.5330293420609837, 0.5421386062012148, 0.5093869690957228, 0.5582149427857186, 0.518191347387065, 0.5173317825623598, 0.5416469795033976, 0.5084596504058045, 0.5421918785042775, 0.5477422514380247, 0.5310119512925238, 0.5643466292817587, 0.5242781494636856, 0.5155404154527214, 0.5291482033929307, 0.5382456754087901, 0.5356590750748988, 0.5146093507692898, 0.5155331068513417, 0.5107845590211122, 0.49709259512648174, 0.5406612656822332, 0.5254527738864513, 0.47365025914289405, 0.5026863892137987, 0.4847454424805501, 0.5006272503882773, 0.5183611209288285, 0.53092180987007, 0.5157132551476162, 0.46251561963104726, 0.5169293299594376, 0.5309439320704129, 0.5391989852294181, 0.4790158079411389, 0.47122792124183543, 0.4971132781015867, 0.46910487097476744, 0.48751167397005046, 0.5255554109525316, 0.5162412951310545, 0.5130572639403853, 0.4942428818535517, 0.5328957713338711, 0.49342435818557, 0.5501664237759702, 0.5197693460251525, 0.5201362742454968, 0.5237561456152058, 0.4652748815906245, 0.5622577950012179, 0.48823986865963215, 0.5287415667309192, 0.5325139497443911, 0.5234965402271439, 0.5042109577886872, 0.5231818403082908, 0.5250771010367166, 0.5360762124308072, 0.5595198982176526, 0.5390296720428642, 0.5173958558385037, 0.5544420060232276, 0.49951493841862527, 0.5337989653060445, 0.5095785699826192, 0.5092820707619263, 0.5029780392806666, 0.5247595247716985, 0.48326734126400933, 0.5169195854876492, 0.5177421568075882, 0.522221419169384, 0.48629807924273805, 0.5435889480897533, 0.5002909464128749, 0.5439988684769034, 0.5353150731956541, 0.5051840045115173, 0.5044098345305791, 0.5412456255681947, 0.4952749628276713, 0.5340410675552362, 0.5135354706285116, 0.4941188701482099, 0.516197614910827, 0.4943104187975763, 0.5158105765325246, 0.5374491969598896, 0.5155184181116019, 0.4966878026009883, 0.5138053353510883, 0.5105627194082454, 0.5122610132401493, 0.5396241117043163, 0.5134566878349844, 0.5259401384015221, 0.5182815964709648, 0.5285885752007355, 0.516177957163346, 0.5563334759714598, 0.5127107475271667, 0.4874713133648201, 0.5244067903285966, 0.5021250919752334, 0.5044661928870448, 0.5267338405837731, 0.47953688262204086, 0.48650159095276946, 0.5392964556524353, 0.5142963971777729, 0.5063975472654332, 0.5600607968181754, 0.40554507338663204, 0.5399376588700816, 0.545017621671174, 0.5616265828004836, 0.4973767064281748, 0.5430904969363628, 0.5189379580423024, 0.5105850217881783, 0.521480167752785, 0.5534426667756333, 0.5153956423314263, 0.5335194069541042, 0.5169605189390077, 0.4999029866865703, 0.507525933957097, 0.5233676633281619, 0.5552967872267145, 0.5094674330613511, 0.5129241220266225, 0.5110691023089866, 0.4584335746726024, 0.5312281888899018, 0.5254136396398952, 0.5165446650942788, 0.5309557545463351, 0.5048137428276306, 0.5117114488525227, 0.5198531396420835, 0.5394069851933558, 0.5076281968592289, 0.5207974047275391, 0.5105043090581345, 0.4931604602345991, 0.511477900346403, 0.5019044410134827, 0.5345194490541979, 0.47867755942119583, 0.503443235735235, 0.5296642343347077, 0.54945776766572, 0.4912799905437545, 0.5011045073585976, 0.525744508067873, 0.5217330724384095, 0.5189579095884409, 0.5115756456910872, 0.5451511500151199, 0.535896804984731, 0.5163639088121765, 0.5443497780139324, 0.5273723704044151, 0.496779477371275, 0.4917025738772586, 0.5199129783600476, 0.5555574253661081, 0.5161268017127224, 0.5048607499557476, 0.5477398977663905, 0.546530762568256, 0.5156703241705856, 0.5047201210239146, 0.5160679440499416, 0.5136941881360079, 0.5355523559374173, 0.5299333707553257, 0.5227759577871374, 0.5201712554338476, 0.5223510136281685, 0.5110540669670143, 0.5695843131945056, 0.49161832166677416, 0.4961044876299475, 0.49763552874042943, 0.4908993688210786, 0.6203475355173481, 0.5273007518863572, 0.5179071592253501, 0.48870424650998145, 0.4915291188781431, 0.5006152076098543, 0.5051876937138088, 0.549873495374664, 0.5099139192307427, 0.5352016596151269, 0.5393076521655412, 0.5292198077217237, 0.5019850116929843, 0.5802505601288612, 0.5176018274304184, 0.5006703799713378, 0.561932312147765, 0.5089532161920891, 0.5091700283664642, 0.5322047549531199, 0.5043468738417454, 0.5358531013580201, 0.5380540458028465, 0.5008285587519216, 0.5337433517523071, 0.4823834150733691, 0.4753609680163763, 0.505213266243515, 0.5043887702271013, 0.4914581613172626, 0.5174839562875031, 0.49123518581447423, 0.48689458078802994, 0.5300705186452404, 0.5223956623641726, 0.5406795324284621, 0.49733160609521687, 0.5235010458003306, 0.5647277576262796, 0.5418150913416404, 0.5392548354421176, 0.5574801161530478, 0.5024513738393106, 0.581494167990156, 0.5148494389173506, 0.5044314548274769, 0.5487261226994308, 0.5077303050462885, 0.5309298126710671, 0.5378249963095838, 0.5278581278098913, 0.5352457299931852, 0.5363301991534517, 0.50403793401343, 0.5043750938470741, 0.5174297357707541, 0.5294196842931942, 0.5327278130044004, 0.5477314044577675, 0.5069204792328617, 0.5140778254744429, 0.5538248693194342, 0.5491302682561112, 0.5237509794212087, 0.5075542421311208, 0.5625274711686843, 0.5015130762339004, 0.5034868825887173, 0.54220980147235, 0.5181131112389572, 0.5287628903203192, 0.525440206769447, 0.46776995358073115, 0.512826497295129, 0.519666941133861, 0.4986933070999592, 0.5240291105429565, 0.522714394720878, 0.4878614757162586, 0.5224837885505929, 0.5373297283497578, 0.5421298665455975, 0.5144862523731037, 0.46935354836044685, 0.519900639715507, 0.4955018560921768, 0.4738751837623973, 0.5316376844997219, 0.5081171921495385, 0.4985036409395137, 0.5813003252688489, 0.5335970516339092, 0.4762377935656132, 0.508959585910158, 0.46247205265378977, 0.49868352101841384, 0.4814936466772881, 0.49725868151567165, 0.5271519103035152, 0.5066182448633876, 0.5178647798899572, 0.5117936427842082, 0.46654043623802244, 0.48721704178310116, 0.507132341599944, 0.5199269188195131, 0.5349912008639794, 0.5522615579829419, 0.521813508155085, 0.5196820560779181, 0.5071109649486041, 0.5173873604155579, 0.5008599941075393, 0.5313476878274377, 0.5269907388469899, 0.5317218685103042, 0.49582117997773123, 0.5135099087365357, 0.5081089457547006, 0.5446626119274962, 0.508331942437029, 0.51634664116695, 0.5003604373302991, 0.5213896188214308, 0.5351908337588607, 0.5414204462566297, 0.486522262917087, 0.5277397848240448, 0.49233356626052516, 0.49626789698784296, 0.5349947181927356, 0.49314428618065687, 0.5481904773230659, 0.53338582078151, 0.4541299149735904, 0.4966500415857948, 0.46687688824159934, 0.5151343716369053, 0.5297308002026409, 0.544528322384804, 0.4647680628253037, 0.5131381304515589, 0.5226081415393601, 0.48642304997716923, 0.5197925432902498, 0.5414602531716171, 0.4485241648330725, 0.546395984553928, 0.5081181032265505, 0.5445620310272324, 0.5401858363126719, 0.5190772768255556, 0.5295093334848084, 0.5453860385999613, 0.5625367731494846, 0.5469067521889486, 0.5268340053047192, 0.5444774119220407, 0.5392453766980488, 0.5056918320420108, 0.509006137526291, 0.5033566040842328, 0.5364133865408225, 0.5276526292590944, 0.5067440161700358, 0.5221384856834861, 0.5409719848979027, 0.537957635828568, 0.5525305280727415, 0.5326875553703152, 0.49540302394826524, 0.5221384856834861, 0.5147075667805157, 0.5311082933081624, 0.5617837853274859, 0.5476161140910815, 0.5444464079917254, 0.5367529793967929, 0.5204822626516602, 0.5334974159646104, 0.5067306658286368, 0.5309739529411961, 0.5367713347549908, 0.5410675342455425, 0.48636414512155246, 0.5241517324148117, 0.523562755304132, 0.543477305930754, 0.5158799558268888, 0.5488435633219555, 0.566749166358963, 0.5454672581360803, 0.560639674341599, 0.5676014601147054, 0.5660451298748783, 0.5468567681909714, 0.5733902917388856, 0.5270418812405993, 0.542024685488593, 0.5428409955768585, 0.5283986829570889, 0.5376071047043031, 0.5363685177489006, 0.5072529142324163, 0.5314218429886189, 0.5241143615382206, 0.5585178387736625, 0.5652565197319649, 0.5229651692954868, 0.5377636149562008, 0.5373400990614658, 0.5418454942266329, 0.4925761721122005, 0.46862303008799705, 0.4820417460695753, 0.4932952444092828, 0.4477401204714536, 0.49089306639418273, 0.5121476344094442, 0.4610922558046334, 0.4793555721513635, 0.46524554170848176, 0.4831452959836376, 0.5039719510693839, 0.4934152361071766, 0.4675040052696892, 0.4696945957678569, 0.48686082933775554, 0.4605067141603368, 0.5046646217156759, 0.5082586993400964, 0.5251819293237168, 0.4759260698794043, 0.47591348761704666, 0.5338869509488589, 0.5437260441918148, 0.5291699524692204, 0.5272448531377503, 0.5422562997127539, 0.5422467448169009, 0.4989708692340766, 0.5581890660089012, 0.5331208527398712, 0.5272448531377503, 0.5264390068057563, 0.544781786622548, 0.520299710583869, 0.5423719593454093, 0.542539821079653, 0.5064434419370962, 0.489190522569185, 0.5245254682408085, 0.5367144527878012, 0.5233230508648206, 0.5374750974663265, 0.48823898256496606, 0.491892366316355, 0.49892288838120646, 0.5152459037016667, 0.4565690259391609, 0.5236393884272939, 0.5497249222261744, 0.5139862441266234, 0.5299878716174853, 0.505640350404795, 0.509169754502848, 0.4835164289429462, 0.5628290761145865, 0.5390144841309342, 0.5423786004296367, 0.5298124027583865, 0.5492678655879898, 0.5054819937841291, 0.5306913985940667, 0.5159671827282545, 0.5068309814505092, 0.49010497866841213, 0.5044547115813942, 0.5306743216834167, 0.5275852605970862, 0.5226552908870391, 0.5111864570043149, 0.5272651002720958, 0.5530771020984264, 0.5519808989467343, 0.515523759773654, 0.5139102377860367, 0.5160829674648246, 0.5057550382659064, 0.5344779820833045, 0.5260537339188499, 0.5230807903388552, 0.5332215158944257, 0.5373247108627944, 0.5002310007142173, 0.5357493289105795, 0.5225808292789051, 0.5000855721975435, 0.4949461563953941, 0.4847042010273147, 0.46834710498694454, 0.48258729807824396, 0.5205446251302165, 0.5596404596748658, 0.5195726970957846, 0.5186020562916607, 0.5730535668071013, 0.5599150655568415, 0.5069271808448337, 0.5521830656217588, 0.49293477830953236, 0.5270108762520472, 0.5643530745235538, 0.5388557993699389, 0.5408603353055108, 0.5368586371261899, 0.4873725239135478, 0.5399672267100349, 0.5311244151935683, 0.5759161394415624, 0.5387107068234422, 0.5335026836763102, 0.5097044558823258, 0.5470548996299847, 0.5113651882900527, 0.5296962575699301, 0.511410687039265, 0.5426535949125189, 0.5390961816013029, 0.543620465390442, 0.5233665521080472, 0.5336454786538399, 0.540179867714486, 0.5368862470772311, 0.5097235777775255, 0.5038081619020259, 0.5357750541231927, 0.5126880261736433, 0.5510566086212295, 0.5281397952354469, 0.5660333373656625, 0.5317876243962605, 0.5091969510878691, 0.5622505275285561, 0.5128617032861262, 0.5478590760215976, 0.5164077422561956, 0.5495399427529432, 0.5157317437033778, 0.5288218743585531, 0.5213214394943184, 0.5311944307951493, 0.5542523859003604, 0.5397786665686071, 0.5533315433624021, 0.5350082287453581, 0.5578055871264296, 0.5554390762995755, 0.5113344456886634, 0.5329084134320461, 0.5184586261694852, 0.5223434880357909, 0.5211792747597344, 0.5310021480467941, 0.5687370136716078, 0.5335537415158721, 0.5370978937575269, 0.5427201633185611, 0.5588040098474172, 0.5638353996203584, 0.5381106137140078, 0.5408951460316995, 0.5365039717569087, 0.5376436410897142, 0.5244458536515128, 0.5231673506518284, 0.48203013770255493, 0.5025562734076836, 0.5381575921069345, 0.5234624492496476, 0.49522583736997206, 0.5179149644610684, 0.5410745060589797, 0.550053294169603, 0.5182831269490242, 0.5210712867279623, 0.5388533777012885, 0.509981697091572, 0.4918579429298362, 0.5488352839075217, 0.49016267301045846, 0.5061993853007494, 0.49245038763105053, 0.5169660847779467, 0.5059413563739185, 0.5155809475496712, 0.5097804532187835, 0.5492533481892375, 0.5164680959617346, 0.5305945154403214, 0.4914325734226517, 0.5284097551988698, 0.4987873039529639, 0.5288693872426597, 0.5095926738844374, 0.5315401292804464, 0.5110282983868952, 0.518756647683317, 0.5096431574575824, 0.5489646434519331, 0.4823932052819019, 0.5360964736581157, 0.44444190509738346, 0.5362116181380183, 0.5255198170005686, 0.5505456333449259, 0.5204843244680335, 0.5113874769406312, 0.5021766730313173, 0.4981772054287878, 0.5516964881578543, 0.5359050449342996, 0.5109649295363039, 0.5181431097683489, 0.5109909345690709, 0.5351540196592288, 0.5109788350798662, 0.5502281083994676, 0.5392437653481463, 0.5007324126753494, 0.504189794420203, 0.5005235729671537, 0.5105069050852684, 0.5425039663392659, 0.5149662677015254, 0.4897126277499157, 0.48955835870997705, 0.5011986322668492, 0.46802527206219163, 0.5149717338729094, 0.5137860729610049, 0.5159661217541944, 0.5007759560692919, 0.5014896904420172, 0.5385369175617377, 0.5356685365807599, 0.547431343923604, 0.5371994780732923, 0.4999902150119173, 0.5522369044196911, 0.5093549881689599, 0.5555968289271764, 0.553280153613752, 0.5372742519229434, 0.5481818169815164, 0.5485119737926084, 0.5891548103615634, 0.5599222455261413, 0.5552021486732766, 0.5358606758239377, 0.5620383253074783, 0.5408629821427737, 0.5705795894615499, 0.5613760519765103, 0.5465371300995299, 0.5293845211980256, 0.5216377771390122, 0.5129183257396039, 0.5066803977818528, 0.5325067375981134, 0.561982099366797, 0.5298420687326002, 0.5614475200803777, 0.5036577924300136, 0.5286822779486917, 0.5507274678530009, 0.5418354712070996, 0.5366253488125794, 0.533096974005772, 0.49063306568202997, 0.5490597217987413, 0.5551743974097236, 0.5185189641960106, 0.5476971123237366, 0.5394765343125987, 0.5073871799042197, 0.5022160453204974, 0.5357041153586696, 0.5515608024338632, 0.5410142789962369, 0.5486327943988987, 0.5414529553085579, 0.5808738101724259, 0.544051514081658, 0.5348423109068694, 0.5679258220867935, 0.5713438424392923, 0.5339272763873794, 0.5805980157675199, 0.5906280333461247, 0.5729253481313559, 0.5812655927291949, 0.5313232662799563, 0.5572933173148364, 0.5673253319835205, 0.5221389733570506, 0.5832714047419929, 0.5551664004175774, 0.5202733465264231, 0.5135293431673424, 0.5595170809944556, 0.5475672671573772, 0.5020793436889924, 0.5551664004175774, 0.5348835350609061, 0.5047367466888059, 0.5350157984817001, 0.5250218476386422, 0.4859835894879911, 0.5140077465649195, 0.5248114583042831, 0.5056464490967633, 0.5420273760729398, 0.5496909934304512, 0.5190998973333687, 0.5083015514109048, 0.5384347811870684, 0.5117705112533478, 0.5312658380487194, 0.5601549774509107, 0.5466884476364968, 0.5535032316938021, 0.538772132527112, 0.5229233032848479, 0.5544105060105833, 0.5982554307084178, 0.5576027672762827, 0.5412341901817627, 0.5351258488822236, 0.505236072107357, 0.5476150329755276, 0.5137632543143305, 0.5401729178990339, 0.5258811881453419, 0.48314713894303396, 0.49382534944769113, 0.4993858072151725, 0.5821469946856528, 0.5391011961029016, 0.543895735979059, 0.4757075175487377, 0.5015644378499029, 0.5096526923585296, 0.5842426817320643, 0.5257559863614913, 0.540254671861931, 0.5265275659114237, 0.5320289822148788, 0.5318215975824623, 0.5298480103038412, 0.5377061210038852, 0.5237287459092599, 0.5364375266467012, 0.5440511566911154, 0.5270077549570962, 0.4950694466096197, 0.5627786878584795, 0.568933416009354, 0.5381971271567141, 0.5393825282950443, 0.5359273914797575, 0.5862740074339653, 0.5601589878258063, 0.552626385351319, 0.4750745983695183, 0.5221761306930975, 0.5462951641759708, 0.5179854295493367, 0.48140331990637913, 0.552626385351319, 0.5136733485893787, 0.5378147550994092, 0.5335790870810906, 0.5233200150572073, 0.5036122903951759, 0.5385196809422269, 0.5265365723996591, 0.5147998906831692, 0.49707011580218124, 0.5711915670767136, 0.585446104327707, 0.5456152293022654, 0.555727720621138, 0.5645447957541169, 0.5077268296660364, 0.5404636846215725, 0.5535694483947827, 0.5520234374474566, 0.5373527445265042, 0.5072589660670389, 0.5239284210231587, 0.5339381437384821, 0.5349714387248182, 0.5520234374474566, 0.5080567461229719, 0.5109398772550573, 0.5487510354649034, 0.5667551167840537, 0.5327240805076535, 0.5372274979951015, 0.559211586677373, 0.5485846069637391, 0.5398931402533458, 0.5609177443353225, 0.5322471986708306, 0.5292969819698523, 0.5480000607423198, 0.5306018845664704, 0.5402954374981694, 0.5130650613771588, 0.5372616209209691, 0.536170853562297, 0.5363290835637882, 0.5020680341731457, 0.5020424288521914, 0.5350514386215941, 0.5379172141770505, 0.536170853562297, 0.5281173833552784, 0.508036335376538, 0.5120038904330567, 0.5212128430167013, 0.5148195965512312, 0.540878133031314, 0.5353114030803131, 0.5243702716265535, 0.5119243436017474, 0.5349511175816786, 0.5670220858278803, 0.515595253594288, 0.5308092900797621, 0.5100097839264365, 0.512998452873717, 0.5667789475536464, 0.5118757306669349, 0.522918820385742, 0.5246560943809488, 0.5159545194933529, 0.505426925833766, 0.545523229289784, 0.5379800162010322, 0.5465376227812502, 0.5157053302530687, 0.5148558369653997, 0.549043326062095, 0.5429395068499137, 0.5505527327166824, 0.5543561618265982, 0.5323031226273471, 0.5434829778958045, 0.5439876448677399, 0.5709537956065478, 0.5685290133931253, 0.5374729584250049, 0.5436685267727714, 0.5224850512150422, 0.4584600693755624, 0.524433767269205, 0.5422975779538592, 0.5863734257146637, 0.5281880853061498, 0.5404006126248563, 0.5573210685360558, 0.5697838986360682, 0.5739959171067703, 0.5362743017501277, 0.5454879838345459, 0.5066947339201486, 0.5064832881466421, 0.5734751830731757, 0.5612595905709417, 0.516983069149276, 0.5392459037995628, 0.4931957622187074, 0.5036045614027883, 0.5885219812877207, 0.5367060229405689, 0.5480121137796753, 0.5247324747418624, 0.5198858425994181, 0.5207991779635015, 0.5864381470548476, 0.5198750723823586, 0.5166714640225052, 0.538741570269434, 0.5177837194389525, 0.5411531532876044, 0.5697212904017037, 0.5160649587356837, 0.5739326052995065, 0.5829515499711881, 0.5253885624152823, 0.5023984991267776, 0.579221258965, 0.5487394290061866, 0.5415258203826485, 0.5889902943070067, 0.5479557267743471, 0.5789465299992361, 0.5512784566658251, 0.5248434373266281, 0.552644205043157, 0.55965235247117, 0.5630648322132565, 0.5607948817684445, 0.5206388640137665, 0.5542656566153559, 0.5443489939341368, 0.5424087114917733, 0.5173584049815424, 0.5415522854692915, 0.5265817047662028, 0.5585911385415993, 0.5464701978085325, 0.5359151108937993, 0.4638143739443887, 0.5498395604875785, 0.5544435013290822, 0.5350746787057307, 0.534741735315426, 0.5165872846584554, 0.5197140731536859, 0.5157110142001132, 0.5475481836725948, 0.4718422165555282, 0.4889894644690759, 0.5120479081065528, 0.49858575881942685, 0.5164851787458834, 0.5412308974066345, 0.529696798299537, 0.4879835182759489, 0.49073688239521346, 0.4850337249939007, 0.5243574501561439, 0.5585889640933972, 0.5235233187071238, 0.493964666974908, 0.4858193440533606, 0.48602437910731494, 0.5389522148729329, 0.537735509244293, 0.5483452838593692, 0.49616760676241073, 0.4761363552463688, 0.5345073502364874, 0.5219969277575653, 0.5484335055376819, 0.5356305926232895, 0.5021447258394859, 0.4716882124781983, 0.5153018432159949, 0.5236115258299975, 0.5522049121929873, 0.526557796986364, 0.5208816576710746, 0.5028585292721355, 0.5148424741283937, 0.5734810935908423, 0.5230530767919038, 0.5027391086399253, 0.5188452640093607, 0.4985868224469088, 0.5017059549843152, 0.5498413244255439, 0.5695676473240232, 0.5183355339358061, 0.5247548865161651, 0.5465551198247001, 0.5342137228625944, 0.4945306901222266, 0.5151852839659884, 0.5382465658664789, 0.5013819638373361, 0.47280403007992977, 0.48569832066300966, 0.4616573631495701, 0.5198734329409209, 0.48754225645666954, 0.5151422886635364, 0.4673892043310818, 0.48481870190301424, 0.45440831387755026, 0.5197565570070573, 0.510845074618127, 0.518166372701964, 0.5133048671142368, 0.5059376154939139, 0.4682166591009342, 0.55810987917984, 0.5272341121264686, 0.5221760734242032, 0.448349374834858, 0.504699899528606, 0.5550594549077146, 0.5432773247922055, 0.4741830147804803, 0.5332113889658993, 0.484030705614808, 0.511530809350787, 0.5349978991567713, 0.5377417181421205, 0.48881959020259347, 0.5431294074564842, 0.5467187770625004, 0.5165846891706035, 0.5226682159361228, 0.5461379393895831, 0.5517246861212934, 0.5189631660487855, 0.5344902630753418, 0.5280006598668573, 0.47027347923830454, 0.4853171312759059, 0.5075594251084037, 0.5173107263939825, 0.5206506436684474, 0.5058836505543947, 0.5240265656510106, 0.5123809675741473, 0.5396519724424225, 0.5136350368267526, 0.4913935748785064, 0.49698538867208686, 0.5234391160149304, 0.5142242165405575, 0.4929364835029469, 0.5319411749005682, 0.5505813886934543, 0.5133698635435693, 0.5128403387843824, 0.5281187101790086, 0.5159821846761417, 0.5291031430586641, 0.5375347578159573, 0.5227216734317506, 0.5631958921535993, 0.552122112034015, 0.5727698887232573, 0.5396515764046477, 0.4491902902216377, 0.5118153346552577, 0.5403195017662317, 0.5032001153209089, 0.5186480335240964, 0.5366790019363946, 0.5186853709625618, 0.5312105160350827, 0.5395951899092383, 0.5018722489124854, 0.46051510840739646, 0.5099588163337924, 0.5095402083949445, 0.5198991406761774, 0.4548415223343424, 0.5455397160154662, 0.5212757092263023, 0.5190812052179707, 0.512353116251731, 0.5353354821220461, 0.5199970899237908, 0.5255656269318552, 0.5241403036448313, 0.5250360693869862, 0.46038629260564007, 0.5194887365582579, 0.48477941350602194, 0.5462976178531209, 0.4892433767143166, 0.5309258982588851, 0.502706588796024, 0.49591786825290546, 0.5209385683589229, 0.507303922309348, 0.4886544410071225, 0.5262107824323928, 0.5082055964280767, 0.48447269681859173, 0.4868646411197058, 0.5252985298387305, 0.5173804448086792, 0.506736585136826, 0.5264301591892135, 0.5064998555867468, 0.4934050056180428, 0.5311044804573148, 0.5288136726703276, 0.5091615550640151, 0.5406183484626, 0.5108034221831508, 0.4934557162423784, 0.5329949077834116, 0.4975204724534462, 0.5337025120223968, 0.5393813367477175, 0.5189999244805372, 0.5161824465599787, 0.5294954604193477, 0.5328809521692908, 0.524240645682938, 0.5453219398486441, 0.5096300197428711, 0.5514359002702369, 0.5027163087065631, 0.49648946709479636, 0.5094469296357905, 0.5115781054214857, 0.481363708821022, 0.5500104682585141, 0.5077422141678011, 0.5353847432427291, 0.5021264099036047, 0.48084942731361946, 0.4603564299216354, 0.5313995286854831, 0.5000824828410069, 0.5139949126285518, 0.5284489393187252, 0.5573209668830593, 0.5173264062657661, 0.5309824858443271, 0.5545101918375533, 0.5167464932700907, 0.5284489393187252, 0.548002634334853, 0.5142677198051695, 0.5551714718488664, 0.56725303965404, 0.5118743589575581, 0.5342170232192278, 0.5427713155430129, 0.5266338711912701, 0.5455489427351723, 0.5538946337987678, 0.5315024425349555, 0.514841060600975, 0.5063000217686787, 0.520061189231938, 0.5209898041315688, 0.556982199105838, 0.5198759430305772, 0.5259224675061388, 0.5418216499271907, 0.5266695573365314, 0.5451949950094043, 0.5012398039275495, 0.5479649168297758, 0.5399098725059619, 0.5215182557475041, 0.5254055138557074, 0.4989392249620991, 0.5073238717999005, 0.5601854064494762, 0.4998937530097285, 0.510328060403441, 0.5012829019543584, 0.553853883260964, 0.5480549196246123, 0.5308323461072755, 0.5517824905345532, 0.5131004579662664, 0.4928296755752393, 0.5376708286761981, 0.5282416647586574, 0.5127480239337661, 0.523269186973033, 0.5052224764317367, 0.5030303364442408, 0.5065842075764528, 0.5479535378322287, 0.5101521450068268, 0.5363186715380583, 0.46915535372277406, 0.4958661502092852, 0.516753605997168, 0.5527401416043491, 0.5289948031366174, 0.522063321304157, 0.47139041942101967, 0.5132331764816719, 0.5183428382223401, 0.523062403020849, 0.4979041076575349, 0.5148892372223268, 0.4836853577586337, 0.4924506783316215, 0.49324854477848573, 0.5514291316772667, 0.49994193382879437, 0.5278135663718617, 0.5461785779113799, 0.5402858808919209, 0.44838251423964404, 0.5457264001796427, 0.5337076625864234, 0.5141632577446289, 0.4829737674842413, 0.5347829353865838, 0.5271204404082046, 0.5279113050888196, 0.4902985484614458, 0.5167860075130788, 0.5187474930071823, 0.511566770262255, 0.5183694808334675, 0.4867982028100994, 0.5507269815798196, 0.5210674998590563, 0.5224407098618642, 0.5308014503953176, 0.5520053346675678, 0.5354281458137773, 0.5464256894874367, 0.5271955420909137, 0.5434009410996429, 0.5212254213795178, 0.566448119107485, 0.5612428138355423, 0.5676890802897712, 0.5330334290228791, 0.5483283881716243, 0.531395254439524, 0.5237486773529783, 0.5387224591799181, 0.5301958450517213, 0.5416222740965827, 0.5305270019647463, 0.5108845472049002, 0.5273693802756384, 0.558431426818273, 0.5347538897314186, 0.543705488816378, 0.5178932229123469, 0.5144746086796405, 0.5174487195422316, 0.5494031851216457, 0.5255196328239244, 0.543705488816378, 0.5117506345748264, 0.48125084345320696, 0.5603777878984658, 0.5308810782292814, 0.5064939904021644, 0.5222031408211438, 0.5338544942054786, 0.5233452053162126, 0.4674547284502865, 0.5521758445520322, 0.5384214516497237, 0.5468328094141426, 0.5369630181470528, 0.5070420858425889, 0.4957743574939205, 0.562289716939536, 0.548569026174097, 0.5239877481863271, 0.5075078682737073, 0.4759258882652825, 0.4846459054794683, 0.5426927956001905, 0.5097290205358462, 0.4995767587849612, 0.5080325484168392, 0.5162248140382041, 0.5396146840190399, 0.5258496362996119, 0.5135085081943372, 0.522116234596304, 0.47628636318713574, 0.4943682399483615, 0.5030915850543516, 0.5084487536715683, 0.4878164691168644, 0.522116234596304, 0.5264085151479472, 0.4885124115707241, 0.4905110509438221, 0.5023697424968185, 0.5172437910877453, 0.5016683499725619, 0.5212367435333702, 0.5158296391513618, 0.48720527335324204, 0.538306843144538, 0.5121394998764012, 0.5120800111134716, 0.533672127923975, 0.5100645128683448, 0.5036763523976462, 0.5443239438488063, 0.5453209720484669, 0.5097232612591294, 0.5224803959021181, 0.5051047895654439, 0.5117693917371926, 0.5025529835624553, 0.4885011228978011, 0.5451059579618618, 0.5445929308237425, 0.5366269367533164, 0.5617818305640216, 0.5353881146385213, 0.5285630073595291, 0.5593686777890798, 0.5242256458754025, 0.4867644697261143, 0.5385054065963488, 0.5450752267906316, 0.5093012799604988, 0.5593686777890798, 0.5287843962062946, 0.5027317317160127, 0.5606949445933418, 0.5447724954194412, 0.5060754727188436, 0.5048038104667215, 0.5212885461905026, 0.5136339871205474, 0.5156243490617656, 0.47746063016023726, 0.5255059952584158, 0.5237204716810228, 0.5286978894215836, 0.5252474482441707, 0.4840704402204881, 0.5187771084189725, 0.5077896361208221, 0.5419253980034937, 0.5341593599370258, 0.5235059522424246, 0.526446952678631, 0.5333795204939032, 0.46673197180511194, 0.5160833271344352, 0.540392126349733, 0.5263172338325357, 0.5330526529945019, 0.5573449972124264, 0.5371907481907816, 0.5148588131686717, 0.5271914115267808, 0.5082251563857979, 0.5105465610015569, 0.5258253531124522, 0.5211474253222169, 0.5004439974355376, 0.5130076054103334, 0.4937772832059263, 0.5095921897650854, 0.5366784247552057, 0.5242051031467091, 0.5004649527913082, 0.5092601373310515, 0.5162018108272, 0.47306148757175315, 0.5226935133712819, 0.5014787237717914, 0.5059521102350715, 0.5330676494564495, 0.5062637780071987, 0.5185692853570503, 0.5396005279460407, 0.5411699143561661, 0.5554644722502147, 0.538011527833386, 0.5116854530685717, 0.5549140344467385, 0.5407431424379878, 0.5310167780316433, 0.5554644722502147, 0.5430985809687234, 0.5123231443348238, 0.5602729507864408, 0.5500056039462025, 0.5267652285492614, 0.5359952046179526, 0.5236357528662751, 0.5339799583132974, 0.5241179256865625, 0.5354334724060765, 0.4952778533330453, 0.5291366455374882, 0.5581651610970639, 0.5133850291980719, 0.4967851555560687, 0.5747768079781911, 0.5306820101592519, 0.511302285370766, 0.5332395500436609, 0.5100612887825846, 0.47642675417564384, 0.5741928017076704, 0.5251883819841214, 0.48915730586605843, 0.5200284693253499, 0.48266552929672674, 0.514344398872532, 0.5398193371584775, 0.5020878357346741, 0.5054325075509347, 0.5194073241368229, 0.5025334951942094, 0.48579604050586955, 0.5771663844330412, 0.5202314245690352, 0.535237084976182, 0.5570898228384762, 0.5342358215079938, 0.48183996663180073, 0.5648783015297638, 0.49928421380922827, 0.5223301740492085, 0.5369520068716642, 0.5142748112598037, 0.5516644044677657, 0.5693875938275206, 0.5316006692707886, 0.5315395911683116, 0.5120942841269535, 0.5309649834841365, 0.5637166870205896, 0.5409547361427409, 0.514828013981646, 0.5370025956315898, 0.5032663317627557, 0.5552021787475565, 0.48229107686100015, 0.5511989205412023, 0.5274908391063593, 0.5368072878831274, 0.5372624600484248, 0.5561524168003529, 0.5313246071001152, 0.5845285163249682, 0.5350759436256336, 0.5365146848264789, 0.503359000082863, 0.520368421605971, 0.5284231350676011, 0.5069792918253013, 0.509000860685759, 0.516706536923734, 0.5046903326013485, 0.5284709537943028, 0.5698153649956631, 0.514765452977405, 0.5166168738502854, 0.5002170468940168, 0.503625961959345, 0.497454216335329, 0.47674997345757963, 0.5271542356285481, 0.4694711595441921, 0.5246511145352635, 0.5469548916290777, 0.5169739253730845, 0.5239053167299138, 0.5191965939618667, 0.5474277399804455, 0.5427534345443912, 0.5689618503969206, 0.5615125009262816, 0.5325025653045985, 0.5398902823204028, 0.49544414548707044, 0.5246407416767762, 0.5288628180591456, 0.5294406963289752, 0.4958101806398209, 0.5088757158454655, 0.5297095766737447, 0.5030606243940566, 0.5042337234748164, 0.5057425619509611, 0.5437890868876525, 0.5105811342849984, 0.5419104819268336, 0.514292720502493, 0.5280918683108953, 0.5192757506319354, 0.5189011459691273, 0.5454847475523737, 0.5638123997757125, 0.527666262679553, 0.5014366631290779, 0.5176968011106376, 0.5426506249136431, 0.532628371097758, 0.528408621753477, 0.5416216064937367, 0.5283411379195704, 0.5434931565885728, 0.536312284197378, 0.5508448915859027, 0.5351640338719598, 0.5075323678133605, 0.49587514856059833, 0.5131527432704638, 0.47383990562980083, 0.5586493741698564, 0.5631724720056323, 0.5257554093165484, 0.49251572350514233, 0.5313318712044859, 0.5489701506063227, 0.513888755255625, 0.510324637281536, 0.5388107720852865, 0.49743737229432305, 0.5539424020479587, 0.5473014063431849, 0.594822147810961, 0.5202064523390024, 0.49037271961149265, 0.47609864377088484, 0.47390459735249074, 0.47072498224417525, 0.497612810373092, 0.4926116969249213, 0.5517038541602927, 0.5259672944574048, 0.5278981859076215, 0.5275588080902811, 0.546974759264331, 0.5396859723543246, 0.522592197675252, 0.4867649507233438, 0.5110892934450546, 0.4875041088623232, 0.5148737592689205, 0.5105978778109682, 0.5162916065157259, 0.4714799533921564, 0.4694573225240535, 0.46816560366470517, 0.4790171974088269, 0.5213343572908817, 0.5162268757278422, 0.5097716267363636, 0.493144579063612, 0.524438118408687, 0.5345928563741392, 0.5097132969091105, 0.522119499157049, 0.5041720765764848, 0.48227152002307055, 0.5256502462579844, 0.5468706443053555, 0.47916742646139254, 0.48559603726088, 0.49972592226574153, 0.5239937643567468, 0.5096701085894162, 0.5301838831631214, 0.5148608714682379, 0.5223838687373468, 0.49098142306769976, 0.5154645031753955, 0.5362491703610505, 0.5542762445734721, 0.511014581995302, 0.5278287938142373, 0.5118833258664893, 0.5377134433306111, 0.5181841535488483, 0.9363579709623285, 0.5249337171174795, 0.49145552678323734, 0.5076134104963133, 0.5205894035072909, 0.49991422906642624, 0.5239986559627805, 0.5291394496791464, 0.4873585144550472, 0.5126764467499874, 0.4906077722910619, 0.5261202211594956, 0.5261740655567586, 0.5327220693513479, 0.5310752050590481, 0.5222122844736854, 0.5166194643623072, 0.4967835857677003, 0.5512840693612779, 0.564402941513436, 0.5278131924126224, 0.5112724554724057, 0.5196382756256772, 0.4787378050568337, 0.5676843906866832, 0.5408216280079955, 0.5097281683354729, 0.5143562784855471, 0.5206652593070611, 0.49310458589701894, 0.5068439073871719, 0.4817449867831438, 0.5088811318732968, 0.4909038974464798, 0.5204604629343389, 0.5049057873530535, 0.5031798355854364, 0.5052185896410815, 0.5170487891971598, 0.5005527370405318, 0.4994640581338561, 0.5185975309178835, 0.5554313062832319, 0.5188906481469784, 0.5084773488211853, 0.49084276210740313, 0.5025488702902657, 0.5215893310322037, 0.5160745890673823, 0.4850367368931504, 0.49713088165425334, 0.4908271421381294, 0.4857292712942425, 0.45418044124337986, 0.5348524537427354, 0.5150524430048089, 0.5199489297934461, 0.5605957728709712, 0.5600018787615219, 0.5575890513753389, 0.536122767680895, 0.5257761865936215, 0.5302246781563991, 0.5722602264742355, 0.5373438680030722, 0.5573588211807723, 0.4996057854642072, 0.5496073504861113, 0.5188625161913492, 0.5102183007726184, 0.5224433134874551, 0.5134235684781838, 0.5327202585547111, 0.4796194279528065, 0.5184430733219969, 0.4689687379809318, 0.5172266297990042, 0.5175533573243006, 0.5495368951872575, 0.4850635893932699, 0.5230203384861047, 0.5157711628206649, 0.5694948795490828, 0.4727648018611599, 0.5079769787423406, 0.4990356858360674, 0.5234209720450385, 0.563141722933481, 0.5083179088577169, 0.5381809063029095, 0.5408021009372026, 0.5167561957509569, 0.5198525183146433, 0.5251470221371323, 0.512621649429018, 0.5404137712155934, 0.49518754932986486, 0.5278845943011359, 0.49322772710003265, 0.5289280362558355, 0.49840316297426573, 0.47802164914002016, 0.5062941486586262, 0.530983051783126, 0.4802740138664438, 0.5254822881741018, 0.49862331981026903, 0.4688464545534605, 0.5353216527610619, 0.5091534021601232, 0.5131743176043485, 0.5453778428243117, 0.5225030888136102, 0.4877180787617378, 0.5434759008673595, 0.5006509392242142, 0.5372178713626137, 0.5285490618916375, 0.5130749536082145, 0.529861251667969, 0.5659008346503874, 0.5300846883596906, 0.4760826945940813, 0.47357908640006174, 0.47736703123964114, 0.48947916889061127, 0.4916349578511169, 0.4846806142230959, 0.5272672338890767, 0.5228427401777768, 0.534599614643383, 0.5446527139685687, 0.5221406071536934, 0.4626332865208713, 0.5467637148248887, 0.5702388443409946, 0.5513225224659941, 0.5454756828303907, 0.5554388310265841, 0.4966715671434301, 0.49684740956877305, 0.5003346319026536, 0.49869903498421403, 0.49092336244027746, 0.4587316750996464, 0.5052604072444729, 0.5868032603968348, 0.5475510732663069, 0.568622888561923, 0.5058669330831556, 0.5012215855751255, 0.5399757300376111, 0.5292129611597981, 0.5521650192112911, 0.5394534929350876, 0.49647341244172816, 0.522358355497022, 0.5150498687058753, 0.5007984488646022, 0.5279920671756445, 0.5031317346239401, 0.465893480880324, 0.5267615771934868, 0.48855167221443285, 0.5048783860876164, 0.5051980684754624, 0.5272180423605117, 0.5014204085159797, 0.48939385988701906, 0.5348219758541404, 0.49453126109653234, 0.5092688841454026, 0.49664454374461725, 0.45797584080501674, 0.5031620220487425, 0.5312990174407971]

In [ ]:
len(ipc_scores)

In [ ]:
import numpy as np

In [ ]:
np.mean(ipc_scores)